# Modelo de detecção de latido de cachorro

Este notebook tem como objetivo exportar o modelo de detecção de latidos. Ele utiliza o [AST](https://huggingface-co.translate.goog/docs/transformers/model_doc/audio-spectrogram-transformer?_x_tr_sl=en&_x_tr_tl=pt&_x_tr_hl=pt&_x_tr_pto=tc) pré-treinado no [AudioSet](https://research.google.com/audioset/), desse modo, o modelo já foi treinado para classificar sons do AudioSet, e possui a classe `Bark`, latido.

## Instalação das dependências

In [ ]:
!pip install -q \
    transformers \
    torch \
    torchaudio \
    librosa \
    soundfile \
    onnx \
    onnxruntime \
    onnxscript \
    numpy

## Imports

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import librosa
import soundfile as sf
from transformers import (AutoFeatureExtractor,AutoModelForAudioClassification)

## Carregamento do modelo pré-treinado

In [ ]:
MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

model = AutoModelForAudioClassification.from_pretrained(MODEL_NAME)

model.eval()

print("Número de classes:", model.config.num_labels)

## Identificação da classe Bark

In [ ]:
id2label = model.config.id2label

bark_ids = []

for idx, label in id2label.items():
    if label.lower() == "bark":
        bark_ids.append(int(idx))

print("IDs encontrados para Bark:", bark_ids)

if len(bark_ids) == 0:
    print("\nAlgumas classes disponíveis:")
    for idx, label in list(id2label.items())[:30]:
        print(idx, label)
else:
    BARK_ID = bark_ids[0]
    print("BARK_ID =", BARK_ID)

O resultado obtido foi:

```text
IDs encontrados para Bark: [75]
BARK_ID = 75
```

## Função para carregamento de áudio

> O AST espera áudio em 16 kHz.

In [ ]:
TARGET_SR = 16000

def load_audio(path):
    audio, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    audio = audio.astype(np.float32)
    return audio

## Teste do modelo

In [ ]:
def predict_pytorch(audio_path):
    audio = load_audio(audio_path)

    inputs = feature_extractor(audio, sampling_rate=TARGET_SR, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits

    probabilities = torch.sigmoid(logits)[0]

    bark_probability = probabilities[BARK_ID].item()

    return bark_probability

### Upload de um áudio para teste

In [ ]:
from google.colab import files

uploaded = files.upload()

audio_path = list(uploaded.keys())[0]

print("Arquivo:", audio_path)

### Resultado do teste

In [ ]:
bark_probability = predict_pytorch(audio_path)

print(f"Probabilidade de Bark: {bark_probability:.4f}")
print(f"Porcentagem: {bark_probability * 100:.2f}%")

Saída:

```text
Probabilidade de Bark: 0.7896
Porcentagem: 78.96%
```

## Criação de função binária

O objetivo é transformar a inferência em: `BARK` (latido) e `NON-BARK` (não latido).

O *threshold* foi definido em 0.5, ou seja, a inferência será `BARK` caso tenha 50% ou mais de chance de ser da classe latido; caso contrário, será `NON-BARK`.

In [9]:
THRESHOLD = 0.5

def detect_bark(audio_path, threshold=THRESHOLD):

    probability = predict_pytorch(audio_path)

    if probability >= threshold:
        label = "BARK"
    else:
        label = "NON-BARK"

    return {
        "label": label,
        "probability": probability
    }

Teste:

In [ ]:
result = detect_bark(audio_path)

print(result)

Saída: 
```text
{'label': 'BARK', 'probability': 0.7895749807357788}
```

# Criação do wrapper para ONNX

O objetivo não é exportar o modelo inteiro retornando centenas de classes, mas somente a detecção de `BARK`.

In [ ]:
class BarkDetector(nn.Module):

    def __init__(self, model, bark_id):
        super().__init__()

        self.model = model
        self.bark_id = bark_id

    def forward(self, input_values):

        outputs = self.model(input_values=input_values)

        logits = outputs.logits

        bark_logit = logits[:, self.bark_id]

        bark_probability = torch.sigmoid(bark_logit)

        return bark_probability.unsqueeze(1)

# Preparação do modelo para exportação

In [ ]:
bark_model = BarkDetector(model, BARK_ID)

bark_model.eval()

# Descobrir o tamanho da entrada

In [ ]:
dummy_audio = np.zeros(TARGET_SR * 10, dtype=np.float32)

dummy_inputs = feature_extractor(
    dummy_audio,
    sampling_rate=TARGET_SR,
    return_tensors="pt"
)

input_values = dummy_inputs["input_values"]

print("Shape:", input_values.shape)
print("dtype:", input_values.dtype)

Saída:
```text
Shape: torch.Size([1, 1024, 128])
dtype: torch.float32
```

# Exportar para ONNX

In [ ]:
ONNX_PATH = "/content/bark_detector.onnx"

torch.onnx.export(
    bark_model,
    (input_values,),
    ONNX_PATH,
    input_names=["input_values"],
    output_names=["bark_probability"],
    opset_version=17,
    dynamo=False
)

print("ONNX exportado:")
print(ONNX_PATH)

# Verificar o ONNX

In [ ]:
import onnx

onnx_model = onnx.load(ONNX_PATH)

onnx.checker.check_model(onnx_model)

print("ONNX válido!")

# Redução do tamanho do modelo (quantização)

O modelo ONNX exportado em `float32` é grande (~86M parâmetros do AST). Para reduzir o tamanho sem precisar retreinar, aplicamos **quantização dinâmica pós-treino**, convertendo os pesos para `int8`. Isso reduz o tamanho do arquivo em aproximadamente 4x, com impacto mínimo na acurácia.

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

ONNX_QUANT_PATH = "/content/bark_detector_int8.onnx"

quantize_dynamic(
    model_input=ONNX_PATH,
    model_output=ONNX_QUANT_PATH,
    weight_type=QuantType.QInt8
)

original_size = os.path.getsize(ONNX_PATH) / (1024 * 1024)
quantized_size = os.path.getsize(ONNX_QUANT_PATH) / (1024 * 1024)

print(f"Tamanho original:   {original_size:.2f} MB")
print(f"Tamanho quantizado: {quantized_size:.2f} MB")
print(f"Redução:            {(1 - quantized_size / original_size) * 100:.1f}%")

# Carregar ONNX Runtime

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(
    ONNX_QUANT_PATH,
    providers=["CPUExecutionProvider"]
)

print("ONNX Runtime carregado (modelo quantizado).")

print("\nInputs:")
for inp in session.get_inputs():
    print(
        inp.name,
        inp.shape,
        inp.type
    )

print("\nOutputs:")
for out in session.get_outputs():
    print(
        out.name,
        out.shape,
        out.type
    )

Saída:

```text
ONNX Runtime carregado.

Inputs:
input_values [1, 1024, 128] tensor(float)

Outputs:
bark_probability [1, 1] tensor(float)
```

## Função de pré-processamento para ONNX

In [17]:
def preprocess_audio(audio_path):

    audio = load_audio(audio_path)

    inputs = feature_extractor(
        audio,
        sampling_rate=TARGET_SR,
        return_tensors="np"
    )

    input_values = inputs["input_values"]

    input_values = input_values.astype(
        np.float32
    )

    return input_values

# Inferência usando ONNX

In [18]:
def predict_bark_onnx(
    audio_path,
    threshold=0.5
):

    input_values = preprocess_audio(
        audio_path
    )

    outputs = session.run(
        ["bark_probability"],
        {
            "input_values": input_values
        }
    )

    probability = float(
        outputs[0][0][0]
    )

    label = (
        "BARK"
        if probability >= threshold
        else "NON-BARK"
    )

    return {
        "label": label,
        "probability": probability
    }

# Teste do ONNX

In [ ]:
result = predict_bark_onnx(
    audio_path
)

print("Resultado:")
print(result)

Saída:

```text
Resultado:
{'label': 'BARK', 'probability': 0.7895747423171997}
```

## Comparar PyTorch × ONNX

In [ ]:
pytorch_probability = predict_pytorch(audio_path)

onnx_result = predict_bark_onnx(audio_path)

onnx_probability = onnx_result["probability"]

print(f"PyTorch:           {pytorch_probability:.8f}")

print(f"ONNX (quantizado): {onnx_probability:.8f}")

print(
    f"Diferença: "
    f"{abs(pytorch_probability - onnx_probability):.8f}"
)

A diferença foi pequena:

```text
PyTorch:           0.78957498
ONNX (quantizado): 0.77797902
Diferença: 0.01159596
```

## Baixar o ONNX

In [ ]:
from google.colab import files

files.download(ONNX_QUANT_PATH)

## Salvar configuração

In [ ]:
config = {
    "model": "MIT/ast-finetuned-audioset-10-10-0.4593",
    "task": "dog_bark_detection",
    "sample_rate": 16000,
    "num_mel_bins": 128,
    "max_length": 1024,
    "threshold": 0.5,
    "input_name": "input_values",
    "output_name": "bark_probability",
    "quantization": "dynamic_int8"
}

with open(
    "/content/bark_detector_config.json",
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=4
    )

files.download("/content/bark_detector_config.json")